# BRIDGE: revised-definition zero-shot and few-shot validation

This controlled variant broadens the antibias definition and updates the complementary harmless definition. The wording is an experimental interpretation, not an official BRIDGE annotation guideline. It reuses the baseline model and scoring method; the few-shot section adds the same training demonstrations used in the original experiment.

Run from the IT299 directory on an allocated DeltaAI GPU. This notebook evaluates validation only, with no training or test-set access. Full label-sequence scoring, including EOS, is unchanged. Upload `scripts/bridge_helpers.py` with this notebook.


In [1]:
from pathlib import Path
import importlib.metadata as metadata
import json
import platform
import socket
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import display

ROOT = Path.cwd()
assert (ROOT / "data/processed/manifest.json").exists(), "Open the notebook from the IT299 folder."
assert torch.cuda.is_available(), "This kernel needs an allocated CUDA GPU."
assert torch.cuda.is_bf16_supported(), "This configuration requires BF16 support."
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
PILOT_ROWS = None  # Set to None for the complete validation split.
MAX_INPUT_TOKENS = 4096
torch.manual_seed(SEED)
np.random.seed(SEED)
versions = {name: metadata.version(name) for name in
            ["torch", "transformers", "accelerate", "bitsandbytes", "pandas", "numpy", "scikit-learn"]}
print("Host:", socket.gethostname())
print("GPU:", torch.cuda.get_device_name(0))
print("Python:", platform.python_version())
print(json.dumps(versions, indent=2))

from functools import partial
from scripts.bridge_helpers import (
    LABELS, SYSTEM_PROMPT, REVISED_SYSTEM_PROMPT, sha256, make_candidate_ids, make_prompt_encoder,
    demonstration_permutations, predict as score_candidates,
    save_evaluation as save_run_evaluation,
)


/sw/user/python/miniforge3-pytorch-2.11.0-595/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Host: gh076.hsn.cm.delta.internal.ncsa.edu
GPU: NVIDIA GH200 120GB
Python: 3.12.9
{
  "torch": "2.11.0+cu130",
  "transformers": "4.57.3",
  "accelerate": "1.14.0",
  "bitsandbytes": "0.49.2",
  "pandas": "3.0.3",
  "numpy": "1.26.4",
  "scikit-learn": "1.9.0"
}


## Validation data

The CSV checksum must match the saved split manifest. Sampling uses seed 42 and approximately preserves label proportions. The notebook prints the full-validation and pilot label counts. Setting `PILOT_ROWS = None` evaluates the full validation split.


In [2]:
manifest_path = ROOT / "data/processed/manifest.json"
manifest = json.loads(manifest_path.read_text())
validation_path = ROOT / "data/processed/validation.csv"
assert sha256(validation_path) == manifest["output_sha256"]["validation.csv"], "Validation checksum mismatch."
validation = pd.read_csv(validation_path, dtype=str, keep_default_na=False)
assert set(validation["split"]) == {"validation"}
assert set(validation["bias_label"]) == set(LABELS)
assert validation["sentence_id"].is_unique

if PILOT_ROWS is None:
    pilot = validation.copy()
else:
    assert len(LABELS) <= PILOT_ROWS <= len(validation)
    counts = validation["bias_label"].value_counts().reindex(LABELS)
    quotas = counts / len(validation) * PILOT_ROWS
    sizes = np.floor(quotas).astype(int)
    for label in (quotas - sizes).sort_values(ascending=False).index[:PILOT_ROWS - sizes.sum()]:
        sizes[label] += 1
    pilot = pd.concat([validation.loc[validation["bias_label"].eq(label)].sample(n=int(sizes[label]), random_state=SEED)
                       for label in LABELS])
    pilot = pilot.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Full validation:")
print(validation["bias_label"].value_counts())
print("Evaluation sample:")
print(pilot["bias_label"].value_counts())


Full validation:
bias_label
harmful     768
harmless    585
antibias    504
Name: count, dtype: int64
Evaluation sample:
bias_label
harmful     768
harmless    585
antibias    504
Name: count, dtype: int64


## Model revision and tokenizer

The model and tokenizer use the saved original baseline revision. The revised system prompt is passed explicitly to both zero-shot and few-shot encoders.


In [3]:
output_root = ROOT / "results/bridge_revised_definition"
output_root.mkdir(parents=True, exist_ok=True)
# Reuse the exact model revision from the original baseline; do not resolve a new one.
lock_path = ROOT / "results/bridge_baseline/model_revision.json"
assert lock_path.exists(), "Copy the original model_revision.json before running."
lock = json.loads(lock_path.read_text())
assert lock["model_id"] == MODEL_ID
MODEL_REVISION = lock["revision"]
print("Model revision:", MODEL_REVISION)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

ORIGINAL_SYSTEM_PROMPT = SYSTEM_PROMPT
SYSTEM_PROMPT = REVISED_SYSTEM_PROMPT
encode_excerpt = make_prompt_encoder(tokenizer, system_prompt=SYSTEM_PROMPT)

lengths = [encode_excerpt(text)["input_ids"].shape[1] for text in pilot["text"]]
print(pd.Series(lengths, name="prompt_tokens").describe(percentiles=[0.5, 0.95, 0.99]))
assert max(lengths) <= MAX_INPUT_TOKENS, (
    "A prompt exceeds the input budget. Review lengths and increase the budget explicitly; no examples are silently truncated or dropped."
)


Model revision: aa8e72537993ba99e69dfaafa59ed015b17504d1
count    1857.000000
mean      252.598277
std        50.626082
min       202.000000
50%       236.000000
95%       344.000000
99%       419.080000
max       966.000000
Name: prompt_tokens, dtype: float64


## Quantized model

NF4 and double quantization reduce weight memory. BF16 is the computation dtype. The model is placed entirely on the allocated GPU, with PyTorch SDPA attention. A failed quantized load remains an error rather than silently switching precision.


In [4]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    quantization_config=quantization,
    dtype=torch.bfloat16,
    device_map={"": torch.cuda.current_device()},
    attn_implementation="sdpa",
)
model.eval()
print("Model footprint (GiB):", model.get_memory_footprint() / 2**30)


Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/sw/user/python/miniforge3-pytorch-2.11.0-595/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model footprint (GiB): 1.8720324039459229


## Candidate scoring and timing

Each candidate consists of the label tokens followed by the tokenizer's end-of-answer token. Its score is the sum of conditional token log-probabilities, without length averaging. Softmax normalizes the three candidate scores, and the highest-scoring label becomes the prediction. These probabilities are relative to the three candidates, not calibrated estimates of correctness.

Three validation prompts warm up the GPU. Per-example timing covers tokenization, device transfer, three candidate forward passes, probability calculation, and prediction selection. Candidates are processed one at a time without sharing a prompt cache. Model loading and CSV writing are excluded. Each completed prediction, its probabilities, and its log scores are appended to CSV. Partial rows survive interruption, but execution does not automatically resume.


In [5]:
candidate_ids = make_candidate_ids(tokenizer)
print("candidate token ids:", candidate_ids)

# Bind this run's model explicitly; a training notebook can bind its adapted model.
predict = partial(
    score_candidates, model=model, candidate_ids=candidate_ids,
    prompt_encoder=encode_excerpt, max_input_tokens=MAX_INPUT_TOKENS,
)

for text in pilot["text"].head(3):
    predict(text)
torch.cuda.reset_peak_memory_stats()
run_dir = output_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir()
pilot[["sentence_id", "group_id"]].to_csv(run_dir / "validation_sample_ids.csv", index=False)
run_config = {
    "condition": "small_zero_shot_revised_definition", "prompt_version": "revised_antibias_v1", "original_system_prompt": ORIGINAL_SYSTEM_PROMPT, "split": "validation", "n": len(pilot),
    "model": lock, "dataset_revision": manifest["revision"],
    "validation_sha256": sha256(validation_path), "manifest_sha256": sha256(manifest_path),
    "seed": SEED, "system_prompt": SYSTEM_PROMPT,
    "precision": "4-bit NF4 + double quantization, BF16 compute", "attention": "sdpa",
    "max_input_tokens": MAX_INPUT_TOKENS,
    "truncation": "none; assert prompt fits", "batch_size": 1,
    "prediction_method": "candidate_label_scoring",
    "candidate_token_ids": candidate_ids,
    "answer_end_token_id": tokenizer.eos_token_id,
    "score_rule": "sum of conditional log probabilities, including answer end",
    "normalization": "softmax over the three candidate scores",
    "length_normalization": False,
    "candidate_forward_passes_per_example": len(LABELS),
    "gpu": torch.cuda.get_device_name(0), "versions": versions,
    "python": platform.python_version(), "status": "running",
}
(run_dir / "run_config.json").write_text(json.dumps(run_config, indent=2) + "\n")
rows = []
for i, example in enumerate(pilot.to_dict("records")):
    prediction = predict(example["text"])
    row = {**example, **prediction}
    rows.append(row)
    pd.DataFrame([row]).to_csv(run_dir / "predictions.csv", mode="a", header=(i == 0), index=False)
    if (i + 1) % 20 == 0 or i + 1 == len(pilot):
        print(f"Completed {i + 1}/{len(pilot)}")
results = pd.DataFrame(rows)
print("Saved:", run_dir)


candidate token ids: {'harmful': [71, 2178, 1262, 151645], 'harmless': [71, 2178, 1717, 151645], 'antibias': [517, 579, 3473, 151645]}
Completed 20/1857
Completed 40/1857
Completed 60/1857
Completed 80/1857
Completed 100/1857
Completed 120/1857
Completed 140/1857
Completed 160/1857
Completed 180/1857
Completed 200/1857
Completed 220/1857
Completed 240/1857
Completed 260/1857
Completed 280/1857
Completed 300/1857
Completed 320/1857
Completed 340/1857
Completed 360/1857
Completed 380/1857
Completed 400/1857
Completed 420/1857
Completed 440/1857
Completed 460/1857
Completed 480/1857
Completed 500/1857
Completed 520/1857
Completed 540/1857
Completed 560/1857
Completed 580/1857
Completed 600/1857
Completed 620/1857
Completed 640/1857
Completed 660/1857
Completed 680/1857
Completed 700/1857
Completed 720/1857
Completed 740/1857
Completed 760/1857
Completed 780/1857
Completed 800/1857
Completed 820/1857
Completed 840/1857
Completed 860/1857
Completed 880/1857
Completed 900/1857
Completed 920/

## Classification results

Accuracy and per-class precision, recall, and F1 compare the highest-scoring label with the reference label. Macro-F1 averages F1 across the three task labels. The confusion matrix uses those same three labels for rows and columns.

The majority baseline always predicts the most frequent class in the saved training-split counts. Subgroup macro-F1 also averages all three labels, with undefined class scores set to zero. Subsets missing classes therefore require care in interpretation. All results describe validation performance, not final test performance.


In [6]:
save_evaluation = partial(
    save_run_evaluation,
    train_label_counts=manifest["label_counts"]["train"],
    device=model.device,
)
zero_shot_metrics = save_evaluation(results, run_dir, run_config)


Candidate probabilities — first 20 examples:


,sentence_id,bias_label,probability_harmful,probability_harmless,probability_antibias,prediction
0,1,harmful,0.9959,0.0036,0.0006,harmful
1,4,harmful,0.5616,0.4374,0.0010,harmful
2,6,harmful,0.0067,0.9931,0.0002,harmless
3,7,harmless,0.0601,0.9399,0.0000,harmless
4,11,harmful,0.9995,0.0005,0.0000,harmful
5,20,harmful,0.0000,0.9996,0.0004,harmless
6,25,harmful,0.9996,0.0002,0.0003,harmful
7,41,harmful,0.8176,0.1824,0.0000,harmful
8,43,harmful,0.9914,0.0086,0.0000,harmful
9,69,harmful,0.5993,0.0232,0.3775,harmful


n                                                                          1857
accuracy                                                               0.570275
macro_f1                                                               0.571274
median_latency_seconds                                                 0.213361
p95_latency_seconds                                                    0.253423
serial_examples_per_second                                             4.596917
peak_allocated_gib                                                     2.048196
peak_reserved_gib                                                      2.945312
majority_label                                                          harmful
majority_baseline             {'n': 1857, 'accuracy': 0.4135702746365105, 'm...
Name: value, dtype: object

,label,precision,recall,f1,support
0,harmful,0.852349,0.496094,0.627160,768
1,harmless,0.473381,0.562393,0.514062,585
2,antibias,0.488112,0.692460,0.572600,504


prediction,harmful,harmless,antibias
bias_label,,,
harmful,381,238,149
harmless,39,329,217
antibias,27,128,349


,field,value,n,accuracy,macro_f1,classes_present
0,is_augmented,no,1732,0.558891,0.556765,3
1,is_augmented,yes,125,0.728000,0.301207,2
2,source,Comparative Agendas Project (CAP),127,0.874016,0.310924,1
3,source,GovReport Summarization,118,0.364407,0.194805,3
4,source,LSC Eviction Laws Database,32,0.781250,0.513504,2
5,source,Mapping Inequality: Redlining Data,168,0.511905,0.507338,3
6,source,On the Books: Jim Crow Laws,210,0.490476,0.500058,3
7,source,Pile of Law,711,0.486639,0.410922,3
8,source,Social Bias Frames,491,0.702648,0.493021,3
9,register,formal,1366,0.522694,0.476780,3


## Few-shot demonstration permutations

Three training examples, one per label, produce six possible orders. Each example stays paired with its correct label. This function only changes demonstration order; it does not select examples or run the model.


In [7]:
# Imported from scripts.bridge_helpers; shared with adapter evaluation.


## Fixed training demonstrations

Seed 43 is selected by default. Change `DEMONSTRATION_SEEDS` to `[44]` or `[45]` for subsequent jobs. The original training checksum, sorted sampling rule, and seeds reproduce the earlier demonstration sets. Each seed has one training example per label and six orders. Labels are unchanged; only the system prompt definition differs from the original experiment.


In [8]:
train_path = ROOT / "data/processed/train.csv"
assert sha256(train_path) == manifest["output_sha256"]["train.csv"], "Training checksum mismatch."
train = pd.read_csv(train_path, dtype=str, keep_default_na=False)
assert set(train["split"]) == {"train"}
assert set(train["sentence_id"]).isdisjoint(validation["sentence_id"])
assert set(train["group_id"]).isdisjoint(validation["group_id"])

DEMONSTRATION_SEEDS = [45]
assert len(set(DEMONSTRATION_SEEDS)) == len(DEMONSTRATION_SEEDS)
demonstration_sets = {}
for demonstration_seed in DEMONSTRATION_SEEDS:
    demonstrations = [
        train.loc[train["bias_label"].eq(label)].sort_values("sentence_id")
        .sample(n=1, random_state=demonstration_seed).iloc[0].to_dict()
        for label in LABELS
    ]
    demonstration_sets[demonstration_seed] = demonstrations
    print("Demonstration seed:", demonstration_seed)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(demonstrations)[["sentence_id", "bias_label", "text"]])
print("Total evaluation runs:", 6 * len(demonstration_sets))


Demonstration seed: 45


,sentence_id,bias_label,text
0,15979,harmful,"Board of directors(a) Composition and Terms.-(1) The Amtrak Board of Directors (referred to in this section as the ""Board"") is composed of the following 10 directors, each of whom must be a citizen of the United States:(A) The Secretary of Transportation."
1,3247,harmless,"2.FindingsCongress finds the following:\n\t\t\t(1)The number of older individuals in the\n\t\t\t United States who are abused, neglected, or exploited is increasing, and a\n\t\t\t large percentage of elder abuse cases are not reported to Federal and State law\n\t\t\t enforcement authorities."
2,17005,antibias,"Distance from public transportation in the east portion, alongside several families of various backgrounds. In the west portion lie several older-type houses."


Total evaluation runs: 6


## Few-shot prompt construction

Each training excerpt is a user message followed by its reference label as an assistant message. The validation excerpt is the final user message. `predict()` accepts this encoder explicitly; its default remains the zero-shot encoder. All six orders must fit the input budget before evaluation begins.


In [9]:
few_shot_jobs = []
for demonstration_seed, demonstrations in demonstration_sets.items():
    for order_number, order in enumerate(demonstration_permutations(demonstrations), start=1):
        encoder = make_prompt_encoder(tokenizer, order, system_prompt=SYSTEM_PROMPT)
        prompt_lengths = [encoder(text)["input_ids"].shape[1] for text in pilot["text"]]
        print(f"Seed {demonstration_seed}, order {order_number}: max tokens = {max(prompt_lengths)}")
        assert max(prompt_lengths) <= MAX_INPUT_TOKENS, (
            f"Seed {demonstration_seed}, order {order_number} exceeds the token budget; no truncation."
        )
        few_shot_jobs.append((demonstration_seed, order_number, order, encoder))


Seed 45, order 1: max tokens = 1148
Seed 45, order 2: max tokens = 1148
Seed 45, order 3: max tokens = 1148
Seed 45, order 4: max tokens = 1148
Seed 45, order 5: max tokens = 1148
Seed 45, order 6: max tokens = 1148


## Evaluate revised-definition few-shot prompts

Run All performs revised-definition zero-shot followed by all six orders for the selected seed. A new timestamped directory under `results/bridge_revised_definition_few_shot` saves each order independently. The original few-shot and revised zero-shot results remain separate. More than one selected seed adds six evaluations per seed.


In [10]:
few_shot_root = ROOT / "results/bridge_revised_definition_few_shot"
experiment_dir = few_shot_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
experiment_dir.mkdir(parents=True)
pd.DataFrame([
    {"demonstration_seed": seed, **example}
    for seed, examples in demonstration_sets.items() for example in examples
]).to_csv(experiment_dir / "demonstrations.csv", index=False)
(experiment_dir / "experiment_config.json").write_text(json.dumps({
    "demonstration_seeds": DEMONSTRATION_SEEDS,
    "prompt_version": "revised_antibias_v1",
    "system_prompt": SYSTEM_PROMPT,
    "orders_per_seed": 6,
    "expected_runs": len(few_shot_jobs),
    "zero_shot_run": str(run_dir.relative_to(ROOT)),
    "selection_policy": "fixed random seeds; no selection based on validation scores",
}, indent=2) + "\n")
few_shot_summaries = []

for demonstration_seed, order_number, order, encoder in few_shot_jobs:
    order_dir = experiment_dir / f"seed_{demonstration_seed}" / f"order_{order_number:02d}"
    order_dir.mkdir(parents=True)
    order_config = {
        **run_config,
        "condition": "small_few_shot_revised_definition",
        "status": "running",
        "shots": len(order),
        "order_number": order_number,
        "training_sha256": sha256(train_path),
        "demonstration_selection": "one per label; sort sentence_id then sample with seed",
        "demonstration_seed": demonstration_seed,
        "demonstration_ids": [example["sentence_id"] for example in order],
        "demonstration_labels": [example["bias_label"] for example in order],
        "prompt_format": "system, three user/assistant demonstration pairs, validation user, assistant prefix",
    }
    (order_dir / "run_config.json").write_text(json.dumps(order_config, indent=2) + "\n")
    pilot[["sentence_id", "group_id"]].to_csv(order_dir / "validation_sample_ids.csv", index=False)
    for text in pilot["text"].head(3):
        predict(text, prompt_encoder=encoder)
    torch.cuda.reset_peak_memory_stats()
    order_rows = []
    for i, example in enumerate(pilot.to_dict("records")):
        row = {**example, **predict(example["text"], prompt_encoder=encoder)}
        order_rows.append(row)
        pd.DataFrame([row]).to_csv(order_dir / "predictions.csv", mode="a", header=(i == 0), index=False)
        if (i + 1) % 20 == 0 or i + 1 == len(pilot):
            print(f"Seed {demonstration_seed}, order {order_number}/6: completed {i + 1}/{len(pilot)}", flush=True)
    order_metrics = save_evaluation(pd.DataFrame(order_rows), order_dir, order_config)
    few_shot_summaries.append({
        "demonstration_seed": demonstration_seed,
        "order_number": order_number,
        "label_order": " -> ".join(order_config["demonstration_labels"]),
        "accuracy": order_metrics["accuracy"],
        "macro_f1": order_metrics["macro_f1"],
        "median_latency_seconds": order_metrics["median_latency_seconds"],
    })
    pd.DataFrame(few_shot_summaries).to_csv(experiment_dir / "order_metrics.csv", index=False)
print("Saved few-shot experiment:", experiment_dir)


Seed 45, order 1/6: completed 20/1857
Seed 45, order 1/6: completed 40/1857
Seed 45, order 1/6: completed 60/1857
Seed 45, order 1/6: completed 80/1857
Seed 45, order 1/6: completed 100/1857
Seed 45, order 1/6: completed 120/1857
Seed 45, order 1/6: completed 140/1857
Seed 45, order 1/6: completed 160/1857
Seed 45, order 1/6: completed 180/1857
Seed 45, order 1/6: completed 200/1857
Seed 45, order 1/6: completed 220/1857
Seed 45, order 1/6: completed 240/1857
Seed 45, order 1/6: completed 260/1857
Seed 45, order 1/6: completed 280/1857
Seed 45, order 1/6: completed 300/1857
Seed 45, order 1/6: completed 320/1857
Seed 45, order 1/6: completed 340/1857
Seed 45, order 1/6: completed 360/1857
Seed 45, order 1/6: completed 380/1857
Seed 45, order 1/6: completed 400/1857
Seed 45, order 1/6: completed 420/1857
Seed 45, order 1/6: completed 440/1857
Seed 45, order 1/6: completed 460/1857
Seed 45, order 1/6: completed 480/1857
Seed 45, order 1/6: completed 500/1857
Seed 45, order 1/6: completed

,sentence_id,bias_label,probability_harmful,probability_harmless,probability_antibias,prediction
0,1,harmful,0.9993,0.0007,0.0,harmful
1,4,harmful,0.9241,0.0759,0.0,harmful
2,6,harmful,0.8176,0.1824,0.0,harmful
3,7,harmless,0.9999,0.0001,0.0,harmful
4,11,harmful,1.0000,0.0000,0.0,harmful
5,20,harmful,0.0003,0.9997,0.0,harmless
6,25,harmful,0.9998,0.0002,0.0,harmful
7,41,harmful,0.2689,0.7311,0.0,harmless
8,43,harmful,0.9988,0.0012,0.0,harmful
9,69,harmful,1.0000,0.0000,0.0,harmful


n                                                                          1857
accuracy                                                               0.619278
macro_f1                                                               0.608586
median_latency_seconds                                                 0.212739
p95_latency_seconds                                                     0.25067
serial_examples_per_second                                             4.627072
peak_allocated_gib                                                      2.05758
peak_reserved_gib                                                      2.945312
majority_label                                                          harmful
majority_baseline             {'n': 1857, 'accuracy': 0.4135702746365105, 'm...
Name: value, dtype: object

,label,precision,recall,f1,support
0,harmful,0.803598,0.697917,0.747038,768
1,harmless,0.519658,0.519658,0.519658,585
2,antibias,0.512397,0.615079,0.559062,504


prediction,harmful,harmless,antibias
bias_label,,,
harmful,536,144,88
harmless,74,304,207
antibias,57,137,310


...

n                                                                          1857
accuracy                                                               0.598815
macro_f1                                                               0.595066
median_latency_seconds                                                 0.213882
p95_latency_seconds                                                    0.216534
serial_examples_per_second                                             4.673642
peak_allocated_gib                                                      2.05758
peak_reserved_gib                                                      2.945312
majority_label                                                          harmful
majority_baseline             {'n': 1857, 'accuracy': 0.4135702746365105, 'm...
Name: value, dtype: object

,label,precision,recall,f1,support
0,harmful,0.832155,0.613281,0.706147,768
1,harmless,0.485849,0.528205,0.506143,585
2,antibias,0.506870,0.658730,0.572908,504


prediction,harmful,harmless,antibias
bias_label,,,
harmful,471,202,95
harmless,48,309,228
antibias,47,125,332


,field,value,n,accuracy,macro_f1,classes_present
0,is_augmented,no,1732,0.594111,0.584998,3
1,is_augmented,yes,125,0.664000,0.284420,2
2,source,Comparative Agendas Project (CAP),127,0.826772,0.301724,1
3,source,GovReport Summarization,118,0.279661,0.211468,3
4,source,LSC Eviction Laws Database,32,0.875000,0.580297,2
5,source,Mapping Inequality: Redlining Data,168,0.601190,0.612276,3
6,source,On the Books: Jim Crow Laws,210,0.595238,0.559168,3
7,source,Pile of Law,711,0.500703,0.431162,3
8,source,Social Bias Frames,491,0.741344,0.440396,3
9,register,formal,1366,0.547584,0.535209,3


Saved few-shot experiment: /u/dmorsovillo/IT299/results/bridge_revised_definition_few_shot/20261022T225813702433Z


## Revised-definition comparison

The summary compares revised-definition few-shot results with revised-definition zero-shot from this run. Each order and seed is reported separately. Standard deviations describe variation, not confidence intervals. Compare the saved per-order metrics with the original-definition runs for the same seeds and orders to assess the prompt change.


In [11]:
order_summary = pd.DataFrame(few_shot_summaries)
assert len(order_summary) == len(few_shot_jobs), "Some planned runs are incomplete."
assert not order_summary.duplicated(["demonstration_seed", "order_number"]).any()
display(order_summary)
metric_columns = ["accuracy", "macro_f1"]
seed_summaries = []
for seed, frame in order_summary.groupby("demonstration_seed"):
    row = {"demonstration_seed": seed, "orders": len(frame)}
    for metric in metric_columns:
        row.update({
            f"{metric}_mean": frame[metric].mean(),
            f"{metric}_order_std": frame[metric].std(ddof=0),
            f"{metric}_min": frame[metric].min(),
            f"{metric}_max": frame[metric].max(),
        })
    seed_summaries.append(row)
seed_summary = pd.DataFrame(seed_summaries)
display(seed_summary)
seed_summary.to_csv(experiment_dir / "seed_metrics.csv", index=False)
comparison = pd.DataFrame({
    "zero_shot": {key: zero_shot_metrics[key] for key in metric_columns},
    "few_shot_mean": order_summary[metric_columns].mean(),
    "few_shot_std_across_runs": order_summary[metric_columns].std(ddof=0),
    "few_shot_min": order_summary[metric_columns].min(),
    "few_shot_max": order_summary[metric_columns].max(),
})
display(comparison)
comparison.to_csv(experiment_dir / "comparison.csv", index_label="metric")


,demonstration_seed,order_number,label_order,accuracy,macro_f1,median_latency_seconds
0,45,1,harmful -> harmless -> antibias,0.619278,0.608586,0.212739
1,45,2,harmful -> antibias -> harmless,0.619278,0.614610,0.213259
2,45,3,harmless -> harmful -> antibias,0.602046,0.587617,0.213953
3,45,4,harmless -> antibias -> harmful,0.593430,0.585460,0.214248
4,45,5,antibias -> harmful -> harmless,0.607970,0.599361,0.214412
5,45,6,antibias -> harmless -> harmful,0.598815,0.595066,0.213882


,demonstration_seed,orders,accuracy_mean,accuracy_order_std,accuracy_min,accuracy_max,macro_f1_mean,macro_f1_order_std,macro_f1_min,macro_f1_max
0,45,6,0.606803,0.009814,0.59343,0.619278,0.59845,0.010506,0.58546,0.61461


,zero_shot,few_shot_mean,few_shot_std_across_runs,few_shot_min,few_shot_max
accuracy,0.570275,0.606803,0.009814,0.59343,0.619278
macro_f1,0.571274,0.598450,0.010506,0.58546,0.614610
